# Symmetry Sector Catalogue

This notebook replaces the old numbered symmetry scripts and the `AllSymmetryCombinations.jl` summary script. The goal is not to inspect every tiny case by hand, but to build one readable catalogue that shows:

1. which symmetry combinations are compatible,
2. how much each combination reduces the Hilbert space,
3. how sector recombination works in a few representative cases.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics


## Reference model

We use the same nearest-neighbor XXZ chain that the old script set used. Its translation, reflection, spin-flip, and `U(1)` symmetries make it a clean playground for basis construction.

In [ ]:
L = 8
half_filling = L ÷ 2
special_k = (0, L ÷ 2)
bond = spin((1.0, "xx"), (1.0, "yy"), (0.6, "zz"))

function sector_spectrum(; kwargs...)
    B = basis(L = L; kwargs...)
    vals = iszero(size(B, 1)) ? Float64[] : sort(eigvals(Hermitian(trans_inv_operator(bond, 2, B))))
    B, vals
end

function collect_rows(specs)
    rows = NamedTuple[]
    for (label, kwargs) in specs
        B, vals = sector_spectrum(; kwargs...)
        push!(rows, (
            label = label,
            dimension = size(B, 1),
            ground_energy = isempty(vals) ? NaN : vals[1],
        ))
    end
    rows
end

single_specs = [
    ("none", (;)),
    ("N = L/2", (; N = half_filling)),
    ("p = +1", (; p = 1)),
    ("p = -1", (; p = -1)),
    ("z = +1", (; z = 1)),
    ("z = -1", (; z = -1)),
    ("k = 0", (; k = 0)),
    ("k = 1", (; k = 1)),
    ("k = L/2", (; k = L ÷ 2)),
]

two_specs = [
    ("N + p = +1", (; N = half_filling, p = 1)),
    ("N + p = -1", (; N = half_filling, p = -1)),
    ("N + z = +1", (; N = half_filling, z = 1)),
    ("N + z = -1", (; N = half_filling, z = -1)),
    ("N + k = 0", (; N = half_filling, k = 0)),
    ("N + k = 1", (; N = half_filling, k = 1)),
    ("k = 0, p = +1", (; k = 0, p = 1)),
    ("k = 0, p = -1", (; k = 0, p = -1)),
]

three_specs = [
    ("N, k = 0, p = +1", (; N = half_filling, k = 0, p = 1)),
    ("N, k = 0, p = -1", (; N = half_filling, k = 0, p = -1)),
    ("N, k = 0, z = +1", (; N = half_filling, k = 0, z = 1)),
    ("N, k = 0, z = -1", (; N = half_filling, k = 0, z = -1)),
    ("N, p = +1, z = +1", (; N = half_filling, p = 1, z = 1)),
    ("N, p = +1, z = -1", (; N = half_filling, p = 1, z = -1)),
]

single_rows = collect_rows(single_specs)
two_rows = collect_rows(two_specs)
three_rows = collect_rows(three_specs)


## Read the catalogue

For each sector we record the basis dimension and the ground-state energy. The dimension is usually the first thing you care about when planning a calculation; the energy is a quick check that different compatible decompositions are behaving sensibly.

In [ ]:
single_rows, two_rows, three_rows


## A few recombination checks

The real consistency test is whether child sectors reproduce the parent spectrum when merged. We check a few representative decompositions rather than every possible one.

In [ ]:
function recombination_error(ref_kwargs, child_kwargs)
    _, ref_vals = sector_spectrum(; ref_kwargs...)
    vals = Float64[]
    for kw in child_kwargs
        _, child_vals = sector_spectrum(; kw...)
        append!(vals, child_vals)
    end
    isempty(ref_vals) ? 0.0 : norm(sort(vals) - ref_vals)
end

recombination = (
    none_from_parity = recombination_error((;), [(; p = p) for p in (1, -1)]),
    none_from_flip = recombination_error((;), [(; z = z) for z in (1, -1)]),
    nk0_from_parity = recombination_error((; N = half_filling, k = 0), [(; N = half_filling, k = 0, p = p) for p in (1, -1)]),
    nk0_from_flip = recombination_error((; N = half_filling, k = 0), [(; N = half_filling, k = 0, z = z) for z in (1, -1)]),
)
recombination


## Plot the dimension reduction

A bar chart makes it easier to compare how aggressively each symmetry choice shrinks the Hilbert space. Here we only plot a representative subset so that the figure stays readable.

In [ ]:
plot_labels = [row.label for row in vcat(single_rows[1:4], two_rows[1:4], three_rows[1:4])]
plot_dims = [row.dimension for row in vcat(single_rows[1:4], two_rows[1:4], three_rows[1:4])]

try
    using Plots
    bar(plot_labels, plot_dims; xrotation = 45, ylabel = "dimension", label = nothing, title = "Representative sector dimensions", size = (900, 500))
catch err
    @info "Plots not available; returning representative dimensions instead" error = err
    (; labels = plot_labels, dims = plot_dims)
end


## Compatibility rules

The old numbered scripts were useful mainly because they taught the compatibility rules by repetition. Here they are in one place:

- `N + z` is only meaningful at half filling for spin-1/2 systems.
- `k + p` requires `k = 0` or `k = L/2`.
- combinations containing both `k` and `p` inherit that same restriction.

With those rules in mind, one notebook is usually easier to navigate than a directory full of tiny scripts.